# Resultados Finais — Marketing Mix Modeling

Este notebook consolida todos os resultados do projeto de Marketing Mix Modeling em um formato executivo, com os gráficos definitivos e os insights acionáveis.

**Estrutura:**
1. Visão geral do modelo e métricas de qualidade
2. Decomposição da receita por canal
3. ROI e curvas de resposta
4. Otimização de budget — cenários e recomendações
5. Insights acionáveis para o time de marketing

In [1]:
import os
import sys
import warnings

import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.model import CHANNELS, MarketingMixModel
from src.optimizer import (
    BudgetOptimizer,
    current_allocation_from_history,
    run_standard_scenarios,
)
from src.transformations import apply_all_transformations
from src.visualizations import (
    CHANNEL_LABELS,
    plot_adstock_effect,
    plot_budget_optimization_comparison,
    plot_channel_contribution_over_time,
    plot_model_diagnostics,
    plot_response_curves,
    plot_revenue_decomposition_waterfall,
    plot_roi_by_channel,
)

# Pipeline completo
caminho_csv = os.path.join("..", "data", "raw", "marketing_data.csv")
df = pd.read_csv(caminho_csv, parse_dates=["date"])
df_t = apply_all_transformations(df)
mmm = MarketingMixModel().fit(df)
contributions = mmm.get_channel_contributions()
cenarios = run_standard_scenarios(mmm, df, expanded_budget=280_000)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


## 1. Qualidade do Modelo

O modelo de Marketing Mix captura a relação entre investimento em mídia e receita, controlando por sazonalidade, feriados e tendência.

In [2]:
# Métricas-chave do modelo
diag = mmm.get_model_diagnostics()

print("╔══════════════════════════════════════════════════╗")
print("║         MARKETING MIX MODEL — MÉTRICAS          ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  R² (coef. determinação)    {diag['r2']:>8.4f}             ║")
print(f"║  R² ajustado                {diag['adj_r2']:>8.4f}             ║")
print(f"║  MAPE                       {diag['mape']:>7.2f}%             ║")
print(f"║  MAE                    R$ {diag['mae']:>8,.0f}             ║")
print(f"║  RMSE                   R$ {diag['rmse']:>8,.0f}             ║")
print(f"║  Durbin-Watson              {diag['durbin_watson']:>8.3f}             ║")
print(f"║  Observações                {diag['n_obs']:>8d}             ║")
print(f"║  Features                   {diag['n_features']:>8d}             ║")
print("╚══════════════════════════════════════════════════╝")

# Diagnósticos visuais
fig_diag = plot_model_diagnostics(mmm)
fig_diag.show()

╔══════════════════════════════════════════════════╗
║         MARKETING MIX MODEL — MÉTRICAS          ║
╠══════════════════════════════════════════════════╣
║  R² (coef. determinação)      0.8833             ║
║  R² ajustado                  0.8707             ║
║  MAPE                          3.16%             ║
║  MAE                    R$    4,367             ║
║  RMSE                   R$    5,267             ║
║  Durbin-Watson                 2.025             ║
║  Observações                     104             ║
║  Features                         10             ║
╚══════════════════════════════════════════════════╝


## 2. Decomposição da Receita

De onde vem a receita? Qual a contribuição do baseline (vendas orgânicas) vs. cada canal de marketing?

In [3]:
# Waterfall — decomposição da receita
fig_wf = plot_revenue_decomposition_waterfall(contributions)
fig_wf.show()

# Contribuição ao longo do tempo
fig_ct = plot_channel_contribution_over_time(mmm, df)
fig_ct.show()

## 3. ROI e Curvas de Resposta

Qual canal entrega mais retorno por real investido? E onde está o ponto de saturação de cada um?

In [4]:
# ROI por canal
fig_roi = plot_roi_by_channel(contributions)
fig_roi.show()

# Curvas de resposta
fig_rc = plot_response_curves(mmm)
fig_rc.show()

## 4. Efeito do Adstock — Carryover por Canal

Como o investimento de uma semana continua impactando as semanas seguintes? Canais com alto decay (branding) retêm mais efeito ao longo do tempo.

In [5]:
# Adstock de dois canais contrastantes
fig_ads1 = plot_adstock_effect(df, df_t, "content_organic")
fig_ads1.show()

fig_ads2 = plot_adstock_effect(df, df_t, "google_ads")
fig_ads2.show()

## 5. Otimização de Budget

Comparação dos dois cenários de otimização:

In [6]:
# Comparativo dos dois cenários
comp1 = cenarios["realocacao_puro"]["comparativo"]
comp2 = cenarios["budget_expandido"]["comparativo"]
budget_atual = cenarios["budget_atual"]

# Cenário 1
total1 = comp1[comp1["canal"] == "TOTAL"].iloc[0]
print("CENÁRIO 1 — REALOCAÇÃO PURA (mesmo budget)")
print(f"  Budget: R$ {budget_atual:,.0f}/mês")
print(f"  Ganho esperado: R$ {total1['ganho_receita']:+,.0f}/mês ({total1['ganho_receita_pct']:+.1f}%)")
print()

fig_c1 = plot_budget_optimization_comparison(comp1)
fig_c1.update_layout(title="Cenário 1 — Realocação Pura (mesmo budget)")
fig_c1.show()

# Cenário 2
total2 = comp2[comp2["canal"] == "TOTAL"].iloc[0]
print("CENÁRIO 2 — BUDGET EXPANDIDO (R$ 280 mil/mês)")
print(f"  Budget: R$ 280.000/mês (+R$ {280_000 - budget_atual:,.0f})")
print(f"  Ganho esperado: R$ {total2['ganho_receita']:+,.0f}/mês ({total2['ganho_receita_pct']:+.1f}%)")

fig_c2 = plot_budget_optimization_comparison(comp2)
fig_c2.update_layout(title="Cenário 2 — Budget Expandido (R$ 280 mil/mês)")
fig_c2.show()

CENÁRIO 1 — REALOCAÇÃO PURA (mesmo budget)
  Budget: R$ 66,412/mês
  Ganho esperado: R$ +83,099/mês (+13.0%)



CENÁRIO 2 — BUDGET EXPANDIDO (R$ 280 mil/mês)
  Budget: R$ 280.000/mês (+R$ 213,588)
  Ganho esperado: R$ +299,819/mês (+47.0%)


## 6. Insights Acionáveis

### O que o modelo nos diz:

| Insight | Detalhe |
|---------|---------|
| **Baseline domina** | A maior parte da receita vem de vendas orgânicas e efeitos de contexto — marketing amplifica, mas não cria do zero. |
| **Conteúdo Orgânico = maior ROI** | Apesar de receber apenas ~5% do budget, entrega o maior retorno por real investido. Canal subinvestido. |
| **Email Marketing = alto retorno** | Segundo melhor ROI, com custo operacional baixo. Oportunidade clara de escala. |
| **Meta e Google = saturados** | Recebem ~70% do budget mas operam próximos ao platô da curva de saturação. Cada real adicional rende menos. |
| **Realocação > Budget adicional** | O ganho de simplesmente redistribuir é significativo — não é preciso gastar mais para ganhar mais. |

### Recomendações práticas:

1. **Curto prazo**: redistribuir budget de Meta/Google para Email e Orgânico, mantendo os pisos mínimos de cada canal.
2. **Médio prazo**: testar incrementos graduais no budget total, priorizando canais com margem na curva de resposta.
3. **Monitoramento**: recalibrar o modelo trimestralmente e comparar ROI previsto vs. realizado.
4. **Limitações**: o modelo não captura efeitos de marca de longo prazo nem interações cross-channel — decisões estratégicas devem complementar os dados com julgamento de negócio.

---

### Tabela Resumo — Contribuição e ROI por Canal

In [7]:
# Tabela final consolidada
tabela_final = contributions.copy()
tabela_final["canal"] = tabela_final["canal"].map(lambda c: CHANNEL_LABELS.get(c, c))
tabela_final = tabela_final.rename(columns={
    "canal": "Canal",
    "contribuicao_total": "Contribuição Total (R$)",
    "contribuicao_semanal_media": "Contrib. Semanal Média (R$)",
    "contribuicao_pct": "% da Receita",
    "investimento_total": "Investimento Total (R$)",
    "roi": "ROI",
})
tabela_final = tabela_final.drop(columns=["canal_display"], errors="ignore")
tabela_final = tabela_final.round(2)
tabela_final.style.format({
    "Contribuição Total (R$)": "R$ {:,.0f}",
    "Contrib. Semanal Média (R$)": "R$ {:,.0f}",
    "% da Receita": "{:.1f}%",
    "Investimento Total (R$)": lambda x: f"R$ {x:,.0f}" if pd.notna(x) else "—",
    "ROI": lambda x: f"{x:.2f}x" if pd.notna(x) else "—",
}).set_caption("Contribuição e ROI por Canal — Período Completo (104 semanas)")

,Canal,Contribuição Total (R$),Contrib. Semanal Média (R$),% da Receita,Investimento Total (R$),ROI
0,Meta Ads,"R$ 1,900,491","R$ 18,274",13.1%,"R$ 550,761",3.45x
1,Google Ads,R$ 0,R$ 0,0.0%,"R$ 411,786",0.00x
2,LinkedIn Ads,"R$ 515,680","R$ 4,958",3.6%,"R$ 206,559",2.50x
3,Email Marketing,"R$ 942,042","R$ 9,058",6.5%,"R$ 137,794",6.84x
4,Conteúdo Orgânico,"R$ 862,612","R$ 8,294",6.0%,"R$ 68,191",12.65x
5,Baseline,"R$ 10,261,473","R$ 98,668",70.9%,—,—
